# 🔤 Lab 02 — Generative AI & LLMs
**Tokenisation, sampling, structured output, and prompt A/B testing**

---
**Real-world scenario:** Booking.com processes 500M+ guest reviews in 40+ languages.
A single property might have 12,000 reviews. Nobody reads all 12,000.
In this lab you will build the core components of a review intelligence system:
tokenise text, understand how temperature changes generation, and A/B test prompt strategies.

**What you will build:**
1. Understand tokenisation and why it drives cost
2. See how temperature controls creativity vs precision
3. A/B test zero-shot vs few-shot vs chain-of-thought prompting
4. Extract structured data from reviews using Pydantic
5. Build a simple prompt testing harness

**Estimated time:** 50 min | **Level:** Beginner | **Runs on:** Databricks Free Edition (CPU) or Colab

In [ ]:
%pip install -q tiktoken openai transformers torch pydantic pandas

In [ ]:
import os
import json
import tiktoken
import pandas as pd
from pydantic import BaseModel
from typing import Literal, Optional

# Set your OpenAI API key — in Databricks use dbutils.secrets or cluster env vars
# os.environ['OPENAI_API_KEY'] = 'sk-...'   # uncomment and set if needed
print('Imports OK')

## Part 1 — Tokenisation: What the Model Actually Sees

LLMs do not read words — they read **tokens**. A token is roughly 3-4 characters (in English).
Tokens are the unit of cost: GPT-4o charges per 1,000 tokens, not per word.
Tokenisation is also language-dependent: Dutch and German use more tokens per word than English.

In [ ]:
enc_gpt4 = tiktoken.encoding_for_model('gpt-4o')

examples = [
    'The hotel breakfast was excellent.',
    'Het ontbijt in het hotel was uitstekend.',  # same sentence in Dutch
    'unbelievable',
    'un-be-liev-able',
    'Hello world! This is a test of tokenisation.',
    'hypotheekrenteaftrek',   # Dutch: mortgage interest deduction
]

print(f'{'Text':<45} {'Tokens':>8}  Token IDs')
print('-' * 85)
for text in examples:
    tokens = enc_gpt4.encode(text)
    decoded = [enc_gpt4.decode([t]) for t in tokens]
    print(f'{text:<45} {len(tokens):>8}  {decoded}')

print('\n--- Cost Implication ---')
review = 'The location was perfect, right in the city centre. The room was clean but a bit small. Breakfast had a great variety of options. The staff were friendly and helpful. I would definitely stay again.'
tokens = enc_gpt4.encode(review)
cost_input = len(tokens) / 1_000_000 * 2.50   # GPT-4o input: $2.50 / 1M tokens
print(f'Review: {len(tokens)} tokens  |  Input cost: ${cost_input:.6f}')
print(f'1 million such reviews (Booking.com scale): ${cost_input * 1_000_000:.2f} in input tokens alone')

## Part 2 — Temperature: Precision vs Creativity

Temperature controls the randomness of sampling. At temperature=0, the model always picks the most likely next token (deterministic). At temperature=1, sampling is proportional to probabilities. Above 1, lower-probability tokens become more likely — outputs get creative but may lose coherence.

In [ ]:
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from environment

prompt = 'In one sentence, describe what makes Amsterdam special.'
temperatures = [0.0, 0.3, 0.7, 1.0, 1.5]

print(f'Prompt: "{prompt}"\n')
print('=' * 70)

for temp in temperatures:
    # Run twice at each temperature to show variability
    responses = []
    for _ in range(2):
        r = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': prompt}],
            temperature=temp,
            max_tokens=80,
        )
        responses.append(r.choices[0].message.content.strip())

    print(f'\nTemperature = {temp}')
    print(f'  Run 1: {responses[0]}')
    print(f'  Run 2: {responses[1]}')
    print(f'  Same answer: {responses[0] == responses[1]}')

## Part 3 — Prompt Strategy A/B Test

Booking.com needs to classify whether a review mentions specific amenities (gym, pool, breakfast, location).
We test three prompt strategies on the same inputs and compare accuracy and consistency.

In [ ]:
test_reviews = [
    {'review': 'Loved the rooftop pool! Breakfast was a bit basic though.', 'expected': ['pool', 'breakfast']},
    {'review': 'Perfect location, 5 minutes from Central Station. Room was tiny.', 'expected': ['location']},
    {'review': 'Great gym facilities and the staff were amazing. Would return!', 'expected': ['gym']},
    {'review': 'Awful experience. Noisy, dirty, and overpriced.', 'expected': []},
    {'review': 'Excellent breakfast spread every morning. The pool was heated too.', 'expected': ['breakfast', 'pool']},
]

AMENITY_LABELS = ['gym', 'pool', 'breakfast', 'location']

# Strategy A: Zero-shot
ZERO_SHOT = '''Which of these amenities are mentioned in the review: gym, pool, breakfast, location?
Review: {review}
Answer with a comma-separated list, or 'none' if none are mentioned.'''

# Strategy B: Few-shot
FEW_SHOT = '''Identify mentioned amenities (gym, pool, breakfast, location) from hotel reviews.

Review: "The heated pool was fantastic and breakfast had great options."
Answer: pool, breakfast

Review: "5 minutes walk to the Rijksmuseum, perfect position."
Answer: location

Review: "Nothing special to note about facilities."
Answer: none

Review: "{review}"
Answer:'''

# Strategy C: Chain-of-thought
CHAIN_OF_THOUGHT = '''You are analysing hotel reviews for amenity mentions.
Amenities to detect: gym, pool, breakfast, location.

Think step by step:
1. Read the review carefully
2. For each amenity, check if it is explicitly or implicitly mentioned
3. List only the amenities that are clearly referenced

Review: "{review}"
Reasoning: Think through each amenity.
Final answer (comma-separated, or 'none'):'''

def extract_amenities(response_text):
    text = response_text.lower().strip()
    if 'none' in text and len(text) < 10:
        return []
    return [a for a in AMENITY_LABELS if a in text]

results = {}
for strategy_name, template in [('zero-shot', ZERO_SHOT), ('few-shot', FEW_SHOT), ('cot', CHAIN_OF_THOUGHT)]:
    correct = 0
    for item in test_reviews:
        resp = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': template.format(review=item['review'])}],
            temperature=0,
        )
        predicted = set(extract_amenities(resp.choices[0].message.content))
        expected = set(item['expected'])
        if predicted == expected:
            correct += 1
    results[strategy_name] = correct / len(test_reviews)

print('Prompt Strategy Comparison (5 reviews):')
print('-' * 40)
for strategy, accuracy in results.items():
    bar = '█' * int(accuracy * 20)
    print(f'{strategy:12} {bar:<20} {accuracy:.0%}')

## Part 4 — Structured Output with Pydantic

Free-text responses are fragile in production. Using structured output guarantees a validated Python object every time — no parsing, no KeyError, no surprises.

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class ReviewAnalysis(BaseModel):
    sentiment: Literal['positive', 'negative', 'mixed']
    score: int = Field(ge=1, le=10, description='Overall score 1-10')
    amenities_mentioned: List[Literal['gym', 'pool', 'breakfast', 'location', 'cleanliness', 'staff', 'value']]
    key_complaint: Optional[str] = Field(None, description='Main complaint if any')
    key_praise: Optional[str] = Field(None, description='Main praise if any')
    would_return: bool
    one_line_summary: str = Field(max_length=100)

review_text = '''Absolutely loved the rooftop pool and the location could not be better — 
steps from the Vondelpark. However, the breakfast options were very limited for the price 
and our room was not cleaned properly on day two. Staff were apologetic but it slightly 
spoiled an otherwise perfect stay. Would probably try again.'''

response = client.beta.chat.completions.parse(
    model='gpt-4o',
    messages=[
        {'role': 'system', 'content': 'You are a hotel review analyst. Analyse the review accurately.'},
        {'role': 'user', 'content': f'Analyse this hotel review:\n\n{review_text}'}
    ],
    response_format=ReviewAnalysis,
    temperature=0,
)

analysis = response.choices[0].message.parsed
print('Review Analysis (Structured Output):')
print(f'  Sentiment:           {analysis.sentiment}')
print(f'  Score:               {analysis.score}/10')
print(f'  Amenities mentioned: {", ".join(analysis.amenities_mentioned)}')
print(f'  Key praise:          {analysis.key_praise}')
print(f'  Key complaint:       {analysis.key_complaint}')
print(f'  Would return:        {analysis.would_return}')
print(f'  Summary:             {analysis.one_line_summary}')
print(f'\nType of result: {type(analysis).__name__}  (a real Python object — not a string!)')

## Part 5 — Prompt Testing Harness

Never ship a prompt change without testing it on a dataset. This harness runs your prompt across multiple inputs and tracks pass rate — the professional way to iterate.

In [ ]:
TEST_CASES = [
    {'input': 'Filthy room, broken shower, staff were rude.', 'expected_sentiment': 'negative', 'expected_would_return': False},
    {'input': 'Perfect in every way. 10/10 no notes.', 'expected_sentiment': 'positive', 'expected_would_return': True},
    {'input': 'Nice location but overpriced for what you get.', 'expected_sentiment': 'mixed', 'expected_would_return': False},
    {'input': 'Good breakfast. Room was small but adequate. Would go back.', 'expected_sentiment': 'mixed', 'expected_would_return': True},
    {'input': 'Best hotel in Amsterdam. The rooftop pool is incredible.', 'expected_sentiment': 'positive', 'expected_would_return': True},
]

results = []
for case in TEST_CASES:
    resp = client.beta.chat.completions.parse(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': 'Analyse hotel reviews accurately.'},
            {'role': 'user', 'content': f'Analyse: {case["input"]}'}
        ],
        response_format=ReviewAnalysis,
        temperature=0,
    )
    a = resp.choices[0].message.parsed
    sentiment_pass = a.sentiment == case['expected_sentiment']
    return_pass = a.would_return == case['expected_would_return']
    results.append({'input': case['input'][:50], 'sentiment_pass': sentiment_pass, 'return_pass': return_pass, 'predicted_sentiment': a.sentiment})

df = pd.DataFrame(results)
print('Prompt Test Results:')
print(df.to_string(index=False))
print(f'\nSentiment accuracy: {df["sentiment_pass"].mean():.0%}')
print(f'Return prediction accuracy: {df["return_pass"].mean():.0%}')

## ✅ Lab 02 Complete

You have:
- Tokenised text and understood the cost implications at Booking.com scale
- Seen how temperature from 0 to 1.5 changes generation determinism and creativity
- A/B tested zero-shot, few-shot, and chain-of-thought — and measured which wins
- Built structured output with Pydantic — production-grade, no parsing needed
- Built a prompt testing harness you can reuse on any prompt change

**Next:** Lab 03 — Embeddings & Vector Search